# 📘 Bài 3 — Tool use, Streaming & Prompt caching

**Tuần 18**

Ba kỹ thuật biến một *lời gọi API* thành một *sản phẩm*:

| Kỹ thuật | Giải quyết vấn đề |
|---|---|
| **Tool use** | Model không biết dữ liệu của bạn, không tính toán chính xác được |
| **Streaming** | Người dùng phải chờ 20 giây nhìn màn hình trắng |
| **Prompt caching** | Prompt dài lặp lại tốn tiền gấp nhiều lần |

In [ ]:
import os
import json
from pathlib import Path

# Nap .env neu co
try:
    from dotenv import load_dotenv
    for p in (Path(".env"), Path("../../../.env")):
        if p.exists():
            load_dotenv(p)
            break
except ImportError:
    pass

import anthropic

MODEL_RE = "claude-haiku-4-5"        # $1 vào / $5 ra  — mặc định của khoá học
MODEL_MANH = "claude-opus-5"         # $5 vào / $25 ra — chỉ dùng khi so sánh

CO_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic() if CO_KEY else None


def goi(**kwargs):
    """Gọi API nếu có key; nếu chưa có thì in thông báo và trả về None.

    Nhờ hàm này, notebook chạy được cả khi bạn chưa lấy API key —
    bạn vẫn đọc được code và giải thích.
    """
    if not CO_KEY:
        print("⏭  Bỏ qua — chưa có ANTHROPIC_API_KEY (xem SETUP.md mục 5)")
        return None
    return client.messages.create(**kwargs)


def van_ban(response) -> str:
    """Lấy toàn bộ text từ response. KHÔNG dùng response.content[0].text!"""
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


print("anthropic SDK:", anthropic.__version__)
print("API key      :", "đã có ✓" if CO_KEY else "CHƯA CÓ — notebook vẫn chạy được")

## 1. Vì sao cần tool use?

In [ ]:
r = goi(
    model=MODEL_RE,
    max_tokens=200,
    messages=[{"role": "user", "content":
               "Đơn hàng A123 của tôi đang ở đâu? Và 84729 * 3821 bằng bao nhiêu?"}],
)
if r:
    print(van_ban(r))
    print()
    print(f"  84729 * 3821 = {84729 * 3821:,}  <- đáp án đúng")
    print()
    print("""  Hai loại việc model KHÔNG tự làm được:
     1. Dữ liệu nó không có  -> đơn A123 nằm trong CSDL của BẠN
     2. Tính toán chính xác  -> nó dự đoán token, không phải chạy phép nhân""")

## 2. Định nghĩa tool

In [ ]:
# --- Đây là "cơ sở dữ liệu" giả lập của bạn ---
DON_HANG = {
    "A123": {"trang_thai": "Đang vận chuyển", "du_kien": "2026-08-25", "tp": "Hà Nội"},
    "B456": {"trang_thai": "Đã giao",         "du_kien": "2026-08-20", "tp": "Đà Nẵng"},
}


def tra_cuu_don(ma_don: str) -> dict:
    """Hàm THẬT sẽ được gọi. Model không chạy hàm này — BẠN chạy."""
    if ma_don not in DON_HANG:
        return {"loi": f"Khong tim thay don {ma_don}"}
    return {"ma_don": ma_don, **DON_HANG[ma_don]}


def tinh_toan(bieu_thuc: str) -> dict:
    """Chỉ cho phép số và toán tử — KHÔNG bao giờ eval() dữ liệu tự do."""
    if not set(bieu_thuc) <= set("0123456789+-*/(). "):
        return {"loi": "Bieu thuc chua ky tu khong cho phep"}
    try:
        return {"ket_qua": eval(bieu_thuc, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"loi": f"{type(e).__name__}: {e}"}


CAC_HAM = {"tra_cuu_don": tra_cuu_don, "tinh_toan": tinh_toan}
print(tra_cuu_don("A123"))
print(tinh_toan("84729 * 3821"))
print(tinh_toan("__import__('os').system('rm -rf /')"))   # bị chặn

> 🔒 `eval()` trên chuỗi do model sinh ra là một lỗ hổng bảo mật. Ở đây ta lọc ký tự và bỏ `__builtins__`. Trong sản phẩm thật, dùng thư viện parser toán học chứ đừng dùng `eval` — nguyên tắc là **tool chạy với quyền tối thiểu**, vì đầu vào của nó do model quyết định, mà model có thể bị người dùng thao túng.

In [ ]:
TOOLS = [
    {
        "name": "tra_cuu_don",
        "description": (
            "Tra cứu trạng thái đơn hàng theo mã đơn. "
            "Dùng khi khách hỏi về tình trạng, vị trí, hoặc ngày giao của một đơn hàng."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ma_don": {"type": "string",
                           "description": "Mã đơn hàng, ví dụ A123"}
            },
            "required": ["ma_don"],
        },
    },
    {
        "name": "tinh_toan",
        "description": (
            "Tính giá trị một biểu thức số học. "
            "Dùng cho MỌI phép tính cần độ chính xác."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "bieu_thuc": {"type": "string",
                              "description": "Biểu thức, ví dụ '1234 * 56'"}
            },
            "required": ["bieu_thuc"],
        },
    },
]
print(f"Đã định nghĩa {len(TOOLS)} tool")

> 🔑 **`description` là prompt engineering.** Model quyết định gọi tool nào **chỉ dựa vào chuỗi này**. "Tra cứu đơn hàng" thì mơ hồ; "Dùng khi khách hỏi về tình trạng, vị trí, hoặc ngày giao" thì rõ.
>
> Khi model gọi sai tool, phản xạ đầu tiên nên là **sửa description**, không phải đổi model.

## 3. ⭐ Vòng lặp tool use

Đây là mẫu bạn sẽ dùng lại suốt Phase 09.

In [ ]:
def chay_voi_tool(cau_hoi: str, max_vong: int = 5, in_ra: bool = True):
    """Vòng lặp tool use đầy đủ.

    Model KHÔNG chạy tool. Nó chỉ NÓI nó muốn gọi tool nào với tham số gì.
    BẠN chạy, rồi gửi kết quả về. Lặp cho đến khi model thôi đòi tool.
    """
    messages = [{"role": "user", "content": cau_hoi}]

    for vong in range(max_vong):
        r = goi(model=MODEL_RE, max_tokens=1024, tools=TOOLS, messages=messages)
        if r is None:
            return None

        # ⚠️ Luôn append NGUYÊN content, không phải chuỗi text
        messages.append({"role": "assistant", "content": r.content})

        if r.stop_reason != "tool_use":
            if in_ra:
                print(f"  [xong sau {vong} lượt gọi tool]")
            return van_ban(r)

        ket_qua = []
        for khoi in r.content:
            if khoi.type != "tool_use":
                continue
            ham = CAC_HAM[khoi.name]
            try:
                out = ham(**khoi.input)
                loi = False
            except Exception as e:
                out, loi = {"loi": f"{type(e).__name__}: {e}"}, True

            if in_ra:
                print(f"  🔧 {khoi.name}({khoi.input}) -> {out}")

            ket_qua.append({
                "type": "tool_result",
                "tool_use_id": khoi.id,          # PHẢI khớp id
                "content": json.dumps(out, ensure_ascii=False),
                "is_error": loi,
            })

        messages.append({"role": "user", "content": ket_qua})

    return "[Đạt giới hạn số vòng lặp]"


kq = chay_voi_tool("Đơn A123 của tôi đang ở đâu? Và 84729 * 3821 bằng bao nhiêu?")
if kq:
    print()
    print(kq)

**Bốn quy tắc bắt buộc của vòng lặp tool use:**

| # | Quy tắc | Sai thì sao |
|---|---|---|
| 1 | Append **nguyên** `r.content`, không phải `van_ban(r)` | Mất khối `tool_use` → API báo lỗi |
| 2 | `tool_use_id` phải khớp `khoi.id` | Lỗi 400 |
| 3 | Mọi `tool_use` phải có `tool_result` tương ứng | Lỗi 400 |
| 4 | `tool_result` gửi trong message vai trò **`user`** | Lỗi 400 |

> ⚠️ **Luôn có `max_vong`.** Không có nó, một lỗi tool lặp lại có thể tạo vòng lặp vô hạn — và mỗi vòng đều tính tiền.

In [ ]:
# Lỗi trong tool KHÔNG được làm sập vòng lặp — model tự phục hồi
kq = chay_voi_tool("Đơn ZZZ999 đang ở đâu?")
if kq:
    print()
    print(kq)
    print()
    print("""  -> Model nhận is_error, đọc thông báo, và trả lời khách một cách hợp lý.
     Đó là lý do PHẢI bắt exception trong vòng lặp và gửi lỗi VỀ cho model,
     thay vì để chương trình sập.""")

## 4. Streaming — vì sao người dùng cần nó

In [ ]:
import time

if CO_KEY:
    prompt = "Viết đoạn giới thiệu 150 từ về nghề AI Engineer."

    t0 = time.time()
    r = client.messages.create(model=MODEL_RE, max_tokens=600,
                               messages=[{"role": "user", "content": prompt}])
    t_thuong = time.time() - t0

    t0 = time.time()
    t_dau_tien = None
    with client.messages.stream(model=MODEL_RE, max_tokens=600,
                                messages=[{"role": "user", "content": prompt}]) as s:
        for _ in s.text_stream:
            if t_dau_tien is None:
                t_dau_tien = time.time() - t0
        _ = s.get_final_message()
    t_stream = time.time() - t0

    print(f"  Không stream — người dùng chờ  : {t_thuong:5.2f}s")
    print(f"  Có stream    — chữ đầu xuất hiện: {t_dau_tien:5.2f}s")
    print(f"  Có stream    — tổng thời gian   : {t_stream:5.2f}s")
    print()
    print("  Tổng thời gian KHÔNG nhanh hơn. Nhưng CẢM GIÁC chờ giảm mạnh.")
else:
    print("⏭  Bỏ qua — chưa có API key")

```python
with client.messages.stream(model=..., max_tokens=..., messages=[...]) as s:
    for chunk in s.text_stream:
        print(chunk, end="", flush=True)
    tin_cuoi = s.get_final_message()      # đủ usage, stop_reason, content
```

**Ba lý do dùng streaming:**

1. **Trải nghiệm** — thấy chữ chạy ngay thay vì màn hình trắng
2. **Tránh timeout** — request dài (`max_tokens` lớn) có thể vượt thời gian chờ HTTP nếu không stream
3. **Hủy sớm** — người dùng thấy sai hướng thì dừng luôn, khỏi trả tiền phần còn lại

> 📌 `get_final_message()` cho bạn `usage` và `stop_reason` như một response thường — bạn không mất gì khi stream.

## 5. ⭐ Prompt caching — kỹ thuật tiết kiệm tiền lớn nhất

In [ ]:
# Một tài liệu dài, lặp lại ở MỌI câu hỏi
TAI_LIEU = """CHÍNH SÁCH ĐỔI TRẢ CÔNG TY ABC

Điều 1. Thời hạn đổi trả
Khách hàng được đổi trả trong 30 ngày kể từ ngày nhận hàng, với điều kiện
sản phẩm còn nguyên tem mác, chưa qua sử dụng, còn đầy đủ phụ kiện.

Điều 2. Trường hợp không áp dụng
Không áp dụng đổi trả cho: đồ lót, mỹ phẩm đã mở nắp, thực phẩm,
sản phẩm giảm giá trên 50%, sản phẩm đặt riêng theo yêu cầu.

Điều 3. Chi phí vận chuyển
Nếu lỗi do nhà sản xuất, công ty chịu toàn bộ phí vận chuyển hai chiều.
Nếu khách đổi ý, khách chịu phí vận chuyển chiều gửi trả.

Điều 4. Thời gian hoàn tiền
Hoàn tiền trong 7 ngày làm việc kể từ khi nhận được hàng trả về.
Hoàn về tài khoản gốc đã thanh toán.

Điều 5. Bảo hành
Sản phẩm điện tử bảo hành 12 tháng. Sản phẩm thời trang không bảo hành.
""" * 12       # nhân lên cho vượt hẳn ngưỡng 2048 token của Haiku

if CO_KEY:
    n = client.messages.count_tokens(
        model=MODEL_RE, messages=[{"role": "user", "content": TAI_LIEU}]).input_tokens
    print(f"  Tài liệu: {len(TAI_LIEU):,} ký tự = {n:,} token")
else:
    print(f"  Tài liệu: {len(TAI_LIEU):,} ký tự (~{len(TAI_LIEU)//3:,} token ước tính)")

print("""
  Ngưỡng tối thiểu để cache:
     Haiku  : 2048 token
     Sonnet : 1024 token
     Opus   : 1024 token
  Dưới ngưỡng -> cache_control bị BỎ QUA, không báo lỗi. Bạn tưởng đang
  tiết kiệm mà thực ra không.""")

In [ ]:
CAU_HOI = [
    "Tôi mua áo 20 ngày trước, còn tem, đổi được không?",
    "Mỹ phẩm đã mở nắp có trả được không?",
    "Bao lâu thì tôi nhận được tiền hoàn?",
]

if CO_KEY:
    print(f"  {'lượt':>5} {'cache_creation':>15} {'cache_read':>11} {'input':>8}")
    print("  " + "-" * 44)

    for i, ch in enumerate(CAU_HOI, 1):
        r = client.messages.create(
            model=MODEL_RE,
            max_tokens=300,
            system=[
                {"type": "text",
                 "text": "Bạn là trợ lý CSKH. Trả lời dựa HOÀN TOÀN vào chính sách dưới đây."},
                {"type": "text",
                 "text": TAI_LIEU,
                 "cache_control": {"type": "ephemeral"}},   # ← điểm cắt cache
            ],
            messages=[{"role": "user", "content": ch}],
        )
        u = r.usage
        print(f"  {i:>5} {getattr(u, 'cache_creation_input_tokens', 0):>15,} "
              f"{getattr(u, 'cache_read_input_tokens', 0):>11,} {u.input_tokens:>8,}")

    print("""
  Lượt 1: cache_creation lớn  (ghi cache, đắt hơn 25%)
  Lượt 2,3: cache_read lớn    (đọc cache, RẺ HƠN 90%)""")
else:
    print("⏭  Bỏ qua — chưa có API key")
    print("""
  Kết quả mong đợi:
     lượt 1: cache_creation LỚN, cache_read 0        <- ghi cache
     lượt 2: cache_creation 0,   cache_read LỚN      <- trúng cache
     lượt 3: cache_creation 0,   cache_read LỚN      <- trúng cache

  Nếu lượt 2 mà cache_read vẫn bằng 0 -> cache đang TRƯỢT. Xem mục 5 bên dưới.""")

In [ ]:
# Tính bằng con số: 2500 token tài liệu, 20 câu hỏi
TOK = 3000
GIA_VAO = 1.00 / 1e6            # Haiku

khong_cache = 20 * TOK * GIA_VAO
co_cache = TOK * GIA_VAO * 1.25 + 19 * TOK * GIA_VAO * 0.10

print(f"  20 câu hỏi trên cùng tài liệu {TOK:,} token:")
print(f"     Không cache : ${khong_cache:.5f}")
print(f"     Có cache    : ${co_cache:.5f}")
print(f"     Tiết kiệm   : {(1 - co_cache/khong_cache):.0%}")
print()
print("""  Bảng giá cache (nhân với giá token vào):
     Ghi cache 5 phút  : 1.25x
     Ghi cache 1 giờ   : 2.00x
     Đọc cache         : 0.10x   <- đây là chỗ tiết kiệm""")

**Bốn quy tắc của prompt caching:**

| # | Quy tắc |
|---|---|
| 1 | Cache khớp theo **tiền tố** — mọi thứ TRƯỚC `cache_control` phải **giống hệt từng byte** |
| 2 | Đặt phần **tĩnh** (tài liệu, hướng dẫn dài, định nghĩa tool) **TRƯỚC**, phần **động** (câu hỏi) **SAU** |
| 3 | Mặc định TTL 5 phút, gia hạn mỗi lần đọc trúng |
| 4 | Dưới ngưỡng token → **bị bỏ qua im lặng** |

> ⚠️ **Cái bẫy phổ biến nhất:** nhét timestamp, tên người dùng, hay session id vào đầu system prompt. Tiền tố đổi mỗi request → **cache không bao giờ trúng**, và bạn còn trả thêm 25% cho việc ghi cache. Bạn sẽ tưởng caching "không hiệu quả" trong khi thực ra nó chưa từng chạy.
>
> **Cách kiểm tra duy nhất:** in `usage.cache_read_input_tokens`. Nếu nó bằng 0 ở lượt thứ hai, cache của bạn đang trượt.

## 6. Ghép cả ba — một trợ lý CSKH hoàn chỉnh

In [ ]:
def tro_ly_cskh(cau_hoi: str):
    """Tool use + caching + streaming trong một hàm."""
    if not CO_KEY:
        print("⏭  Bỏ qua — chưa có API key")
        return

    system = [
        {"type": "text", "text": "Bạn là trợ lý CSKH của công ty ABC. Trả lời ngắn gọn, thân thiện."},
        {"type": "text", "text": TAI_LIEU, "cache_control": {"type": "ephemeral"}},
    ]
    messages = [{"role": "user", "content": cau_hoi}]

    for _ in range(5):
        r = client.messages.create(model=MODEL_RE, max_tokens=1024,
                                   system=system, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": r.content})

        if r.stop_reason != "tool_use":
            break

        kq = []
        for k in r.content:
            if k.type == "tool_use":
                print(f"  🔧 {k.name}({k.input})")
                kq.append({"type": "tool_result", "tool_use_id": k.id,
                           "content": json.dumps(CAC_HAM[k.name](**k.input),
                                                 ensure_ascii=False)})
        messages.append({"role": "user", "content": kq})

    # Lượt cuối: stream để người dùng thấy chữ chạy
    print("  🤖 ", end="")
    with client.messages.stream(model=MODEL_RE, max_tokens=1024,
                                system=system, messages=messages[:-1]) as s:
        for chunk in s.text_stream:
            print(chunk, end="", flush=True)
        cuoi = s.get_final_message()
    print()
    print(f"\n  [cache_read={getattr(cuoi.usage, 'cache_read_input_tokens', 0):,} token]")


tro_ly_cskh("Đơn A123 tới chưa? Mà nếu tôi muốn trả lại thì mất phí ship không?")

> 📌 Trong sản phẩm thật, bạn thường **không stream các lượt gọi tool** (người dùng không cần thấy) mà chỉ stream **câu trả lời cuối**. Đó là điều hàm trên làm.

## ✍️ Bài tập & Mini-project

Tuần này không có bài tập pytest riêng — kiến thức ở đây đi thẳng vào **mini-project**:

```
curriculum/07-llm-engineering/mini_project/README.md
```

Bạn sẽ xây một trợ lý CSKH có tool use, caching, streaming, **và một bộ eval** để chứng minh nó hoạt động (Tuần 19).

**Trước khi sang bài 4, tự kiểm tra:**

- [ ] Vì sao phải append `r.content` chứ không phải `van_ban(r)`?
- [ ] Ba thứ phải khớp giữa `tool_use` và `tool_result` là gì?
- [ ] Vì sao đặt timestamp ở đầu system prompt làm hỏng cache?
- [ ] Streaming có làm request nhanh hơn không?